# Math 372 · Computational Lab 1
## Spectral counting

**Assigned** Monday, 14 September · **Due** Monday, 28 September, 10 p.m., via Gradescope

---

One idea, in two halves:

> **Use exact arithmetic to check identities. Use floating point to measure asymptotics.**

Both halves are real mathematics and each is wrong in the other's job. By the end you will have
watched a true identity between integers come out wrong by more than a billion, and you will have
measured a growth rate that no exact computation would have handed you.

### How to work through it

- Four cells are marked **YOUR TURN**. Each takes a few lines at most, and the cell just above it
  shows you the pattern. Everything else is given: run it and read it.
- Every YOUR TURN cell has a **prediction** cell before it. Fill the prediction in *before* you
  run anything. This is the point of the lab, not a formality — and a wrong prediction you left
  visible is worth more than a right one you back-filled.
- The YOUR TURN cells end with `raise NotImplementedError(...)`. Delete that line once the cell
  is filled in. **When Restart & Run All completes without error, you are done.**
- This is built to take about **three hours**. If you pass five, stop, write up what you have, and
  email me.

No prior Sage experience is assumed. If a cell you were *given* fails, that is my problem, not
yours — email me.

---
## Setup

Run this first. If it fails, you are on the wrong kernel: **Kernel → Change kernel → SageMath**.

In [ ]:
try:
    matrix
except NameError:
    raise RuntimeError("Not a Sage kernel. Kernel > Change kernel > SageMath.")

def A_of(G):
    "Adjacency matrix of G, over the integers."
    return G.adjacency_matrix()

def _roots(M):
    return M.charpoly().roots(ring=AA, multiplicities=True)

def eigenvalues(M):
    "The eigenvalues of M, exactly, as a plain list (repeated according to multiplicity)."
    return [r for r, m in _roots(M) for _ in range(m)]

def show_spectrum(M, digits=6):
    "Print the exact spectrum, largest first, with a decimal approximation alongside."
    for r, m in sorted(_roots(M), reverse=True):
        print("   %-24s  (%s)   multiplicity %s" % (r, r.n(digits=digits), m))

def walks(M, l):
    "Number of closed walks of length l, i.e. tr(M^l). Exact."
    return (M**l).trace()

def basics(G):
    "The invariants you can compute by hand, to check predictions against."
    M = A_of(G)
    print("vertices            ", G.num_verts())
    print("edges               ", G.num_edges())
    print("degree sequence     ", sorted(G.degree(), reverse=True))
    print("max degree Delta    ", max(G.degree()))
    print("bipartite?          ", G.is_bipartite())
    print("triangles           ", G.triangles_count())
    print("tr A, tr A^2, tr A^3", walks(M,1), walks(M,2), walks(M,3))

def gap_report(G, l=40, name=""):
    "lambda_1, the next largest |eigenvalue|, their ratio, and the ratio test at step l."
    M = A_of(G)
    ev = sorted(eigenvalues(M), reverse=True)
    l1, rest = ev[0], [abs(x) for x in ev[1:]]
    l2 = max(rest) if rest else 0
    est = "undefined" if walks(M, l) == 0 else "%.4f" % RDF(walks(M, l+1) / walks(M, l))
    print("%-12s lambda_1 = %-7.4f |lambda_2| = %-7.4f  ratio = %-6.4f   estimate = %s"
          % (name or G.name(), RDF(l1), RDF(l2), RDF(l2/l1) if l1 != 0 else 0, est))

def K(n): return graphs.CompleteGraph(n)
def C(n): return graphs.CycleGraph(n)
def Q(n): return graphs.CubeGraph(n)

D = Graph({'a': ['b','c','d'], 'b': ['c','d']})
AD = A_of(D)

print("Setup OK.")

---
# Part 0 · Warm-up

The graph $D$ from problem set 1.

In [ ]:
print(AD)
D.plot(vertex_size=600, figsize=4)

### Prediction 0.1

Look at the picture, not the matrix. **How many walks of length 2 are there from $a$ to $b$?**
List them.

*Your prediction:*

In [ ]:
# YOUR TURN
# Compute A^2 and print the entry that counts walks of length 2 from a to b.
# Sage matrices are indexed M[i, j], counting from 0, and the vertices are in
# sorted order: a, b, c, d.   Hint: AD**2 is the matrix A^2.


raise NotImplementedError("delete this line when the cell is filled in")

If the matrix disagrees with your count, the matrix is right, and the walk you missed is the
interesting thing.

---
# Part 1 · Exact

### 1.1 · The families from week 1

You computed these three spectra by hand. Here they are by machine.

In [ ]:
print("C(6):"); show_spectrum(A_of(C(6)))
print("Q(3):"); show_spectrum(A_of(Q(3)))

**Question 1.1.** Compare with what you derived in week 1: $\operatorname{spec}(C_n) =
\{2\cos(2\pi k/n)\}$ and $\operatorname{spec}(Q_n) = \{n-2i \text{ with multiplicity }
\binom{n}{i}\}$. One of the two outputs has fewer *distinct* values than the formula might lead you
to expect. Which, and why?

*Your answer:*

### 1.2 · The first thing that goes wrong

$\operatorname{tr}A^{\ell} = \sum_i \lambda_i^{\ell}$ is an identity between integers: the left
side counts closed walks, so every one of its digits means something.

Two of $D$'s eigenvalues are irrational. Compute the right-hand side from their floating-point
values, for growing $\ell$, and compare with the exact count.

In [ ]:
for l in [4, 20, 40, 60]:
    exact  = walks(AD, l)
    approx = sum(RDF(r)**l for r in eigenvalues(AD))
    print("l = %2d   exact: %-26s  floating point: %s" % (l, exact, approx))

print()
print("at l = 60, both written out as integers:")
print("   exact:", walks(AD, 60))
print("   float:", "%.0f" % sum(RDF(r)**60 for r in eigenvalues(AD)))

**Question 1.2.** At $\ell = 4$ the two agree. By $\ell = 60$ they differ by more than a billion.
Neither the theorem nor Sage is broken. In one or two sentences, say what happened.

Two things to notice on the way. Count the digits: how many does the exact answer have, and how
many does a floating-point number carry? And compute the *relative* error at $\ell=60$ — it is
about $4\times10^{-16}$, which is as good as a double can do. Floating point is not
malfunctioning. It is keeping the promise it makes, and that promise is not what this question
needs.

*Your answer:*

In [ ]:
# YOUR TURN
# Redo the sum WITHOUT floating point -- drop the RDF( ) -- and check with == that it
# equals walks(AD, l) exactly, for l = 4, 20, 40, 60.


raise NotImplementedError("delete this line when the cell is filled in")

---
# Part 2 · Approximate

Now the other half. For large $\ell$ one eigenvalue dominates $\sum_i\lambda_i^{\ell}$, so the
closed-walk counts should grow like $\lambda_1^{\ell}$, and the ratio of consecutive counts
$$\frac{\operatorname{tr}A^{\ell+1}}{\operatorname{tr}A^{\ell}}$$
should converge to $\lambda_1$. This is a question exact arithmetic answers badly — it hands you
an enormous fraction — and floating point answers well.

### Prediction 2.1

What should the ratio converge to for $K_6$? For the Petersen graph? Numbers, please.

*Your predictions:*

In [ ]:
for G, name in [(K(6), "K_6"), (graphs.PetersenGraph(), "Petersen")]:
    M = A_of(G)
    print("%-9s" % name, ["%.4f" % RDF(walks(M, l+1) / walks(M, l)) for l in [10, 20, 30, 40]])

### 2.2 · The second thing that goes wrong

Same test, two more graphs.

In [ ]:
for G, name in [(Q(3), "Q_3"), (C(8), "C_8")]:
    M = A_of(G)
    print("%-9s" % name, ["%.4f" % RDF(walks(M, l+1) / walks(M, l)) for l in [10, 20, 30, 40]])

**Question 2.2.** The test confidently reports $\lambda_1 = 0$ for two graphs whose largest
eigenvalues are $3$ and $2$. What went wrong? You proved the relevant fact on problem set 1 —
say which problem, and what it has to do with this.

*Your answer:*

### 2.3 · The fix

**Prediction.** Change *one thing* in the cell above so that it converges for these graphs. Say
what it will converge to before you run it.

*Your prediction:*

In [ ]:
# YOUR TURN
# Copy the cell from 2.2, make your one change, and run it on Q(3), Q(4) and C(8).
# (If you see ZeroDivisionError, that is information too -- what does it tell you?)


raise NotImplementedError("delete this line when the cell is filled in")

### 2.4 · How fast?

Both tests converge, but not equally fast. For four graphs, here are the largest eigenvalue, the
next largest in absolute value, their ratio, and what the ratio test actually reports at $\ell=40$.

In [ ]:
gap_report(K(6), name="K_6")
gap_report(graphs.PetersenGraph(), name="Petersen")
gap_report(C(7), name="C_7")
gap_report(C(9), name="C_9")

**Question 2.4.** One of the four is still badly wrong at $\ell=40$. Which one, and what number in
the table predicts that it would be? Finish the sentence: *the rate at which the walk counts
reveal $\lambda_1$ is governed by …*

*Your answer:*

You have just measured something with a name. Ask about it in class — it is a row on the poster,
and it is the first row this course has earned from a computation rather than a proof.

---
# Part 3 · A graph of your own

Pick a graph that is **not** $K_n$, $C_n$ or $Q_n$, with **at least 8 vertices**. Some options,
roughly in order of how much they will surprise you:

1. **The Petersen graph**, `graphs.PetersenGraph()`.
2. **A Cayley graph on $\mathbb{Z}_2^n$**: binary strings of length $n$, with $u\sim v$ when $u+v$
   lies in a set $\Gamma$ you choose. (Weight-one vectors give $Q_n$; choose something else.)
3. **A circulant on $\mathbb{Z}_n$**: $i\sim j$ when $i-j$ lies in a set of your choosing.
4. **A graph from something you care about** — a transit map, a molecule, a cast of characters —
   provided you can draw it by hand.

### Prediction 3.1 — three predictions, before computing anything

Using results you now own, write down three things that must be true of your graph's spectrum,
and say where each comes from:

- $\lambda_1 \le \Delta$ *(problem set 1, bonus)*
- bipartite $\Rightarrow$ spectrum symmetric about $0$ *(problem set 1, problem 4)*
- $\operatorname{tr}A^2 = 2\cdot(\text{edges})$, and $\operatorname{tr}A^3 = 6\cdot(\text{triangles})$

*Your three predictions, with reasons:*

1.
2.
3.

In [ ]:
# YOUR TURN
# Build your graph, call it G, and draw it.
# For a graph from a list of edges:  G = Graph([('x','y'), ('y','z'), ...])


raise NotImplementedError("delete this line when the cell is filled in")

In [ ]:
basics(G)
print()
show_spectrum(A_of(G))

**Question 3.2.** Against the output above, does each of your three predictions hold? For each one
that held, say what would have had to be true of the graph for it to fail. If one missed, find
out why — a missed prediction here is the most interesting thing in your lab, and it should get
the most words.

*Your answer:*

---
# Part 4 · Write-up

Prose, in your own words, for another student in this course. About a page in total.

### 4.1 · What you found, and where exact versus approximate mattered

Half a page. Not a narration of what you typed — an account of what is true and how you know it.
Say where in this lab you used exact arithmetic, where you used floating point, and give one
example of a question that the other choice would have answered badly.

*Your write-up:*

### 4.2 · A prediction that was wrong

Name one prediction from this lab that did not survive contact with the computation, and explain
what you had misunderstood. If every prediction held, say so, and instead name the one you were
least sure of and why. **This section is required and graded as content.**

*Your write-up:*

### 4.3 · Methods and tools

Three or four sentences: what software you used and for what, what you checked and against
what, and any help you had — from a classmate, from me, or from a machine. The course AI policy
applies: help with *syntax* is fine, help deciding *what to compute* is not, and every line you
submit should be one you can explain. All prose in this notebook is your own.

*Your note:*

---
### How it is graded

Holistically, on three things: **predictions made before computing and left visible when wrong;
computations checked against something that did not produce them; and prose that says
something.** The most common way to lose points is a clean notebook with empty prediction cells.

### Before you submit

1. **Kernel → Restart Kernel and Run All Cells.** Read the output top to bottom.
2. No `NotImplementedError` remains.
3. Every prediction cell is filled in, including the ones you got wrong.
4. **Export a PDF**: File → Save and Export Notebook As → PDF (or print the page to PDF). Check
   that the outputs are in it.
5. **Submit both files to Gradescope in one submission** — the `.ipynb` and the `.pdf`. I read
   and mark up the PDF; I run the notebook when I want to check something. Gradescope's mobile
   app cannot submit to this assignment, so use a browser.

---
## Optional · if you want to keep going

Not graded. **Look at a random graph:** build `graphs.RandomGNP(200, 0.5)`, compute its
eigenvalues numerically (`A_of(G).change_ring(RDF).eigenvalues()`), and histogram them. One
eigenvalue sits off to the right on its own. The rest do something worth seeing.